In [1]:
%pip install h3 osmium pandas pyarrow scipy networkx scikit-learn joblib folium

In [2]:
%pip install "h3>=4,<5" osmium pandas pyarrow scipy networkx scikit-learn joblib folium

In [3]:
import h3

print("H3 installé :", h3.__version__)
print("Test :", h3.latlng_to_cell(48.6921, 6.1844, 8))

In [4]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_32_pipeline_citiz_decision_ml_optimisation.py"
)

print("Script trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError(
        "Importer le script dans Files."
    )

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)

# Budgets d'exemple configurables
CONFIG["budget_eur"] = 250_000
CONFIG["second_budget_eur"] = 500_000

# Exécuter toutes les étapes
citiz_decision.run("all")


In [5]:

from pyspark.sql import functions as F

plans = spark.table(
    "gold.citiz_32D_scenarios_budget"
)

print("=== SYNTHESE BUDGETAIRE ===")

display(
    plans.groupBy("budget_eur")
    .agg(
        F.count("*").alias("nb_stations"),
        F.round(
            F.sum("cout_1ere_annee_hypothese_eur"), 0
        ).alias("cout_total_eur"),
        F.round(
            F.sum("population_nouvellement_couverte_marginale"), 0
        ).alias("nouveaux_habitants_couverts"),
        F.round(
            F.sum("population_gagnant_autopartage_marginale"), 0
        ).alias("habitants_gagnant_autopartage"),
        F.sum("reservations_mois_scenario_central")
         .alias("reservations_mois_simulees")
    )
    .orderBy("budget_eur")
)

print("=== SITES A VERIFIER POUR LES ARBRES ===")

display(
    plans.filter(
        F.col("distance_arbre_osm_m").isNull()
        | (F.col("distance_arbre_osm_m") < 15)
    )
    .select(
        "budget_eur",
        "rang",
        "candidate_id",
        "distance_arbre_osm_m",
        "cout_1ere_annee_hypothese_eur"
    )
    .orderBy("budget_eur", "rang")
)

print("=== GENERALISATION DES PREDICTIONS ML ===")

display(
    spark.table("gold.citiz_32C_predictions")
    .groupBy("hors_distribution_population")
    .count()
)


In [2]:
%pip install folium

In [1]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_33_citiz_securite_arbres_couts_optimisation.py"
)

print("Script trouvé :", script.exists())

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)

# Budgets à comparer
CONFIG33["budgets_eur"] = [
    250_000,
    500_000,
    750_000
]

# Préférence Citiz : éviter les arbres proches
CONFIG33["interdire_arbre_osm_proche"] = True
CONFIG33["distance_arbre_seuil_m"] = 15

# Coût estimé sur la première année
CONFIG33["budget_mesure"] = "premiere_annee"

# Lancer l'ensemble de l'étape 33
citiz_33.run("all")


In [3]:

import folium

print("Folium installé :", folium.__version__)

# Vérifier que l'optimisation est déjà enregistrée
assert spark.catalog.tableExists(
    "gold.citiz_33C_scenarios_budget"
), "La table de résultats 33C est introuvable"

# Recharger le script si la session Python a redémarré
if "citiz_33" not in globals():
    from pathlib import Path

    script = Path(
        "/lakehouse/default/Files/"
        "etape_33_citiz_securite_arbres_couts_optimisation.py"
    )

    exec(
        compile(
            script.read_text(encoding="utf-8"),
            str(script),
            "exec"
        )
    )

# Reprendre UNIQUEMENT la génération du dashboard
citiz_33.run("33d")


In [4]:

from pyspark.sql import functions as F

kpi = spark.table(
    "gold.citiz_33D_dashboard_kpi"
)

print("=== RESULTATS FINAUX CITIZ ===")

display(
    kpi.orderBy("budget_eur", "territoire")
)

print("=== TOP 10 DES STATIONS RECOMMANDEES ===")

display(
    spark.table("gold.citiz_33C_scenarios_budget")
    .filter(F.col("budget_eur") == 500000)
    .select(
        "rang",
        "nom_commune",
        "population_desservie_5min",
        "population_nouvellement_couverte_marginale",
        "population_gagnant_autopartage_marginale",
        "cout_1ere_annee_eur",
        "distance_arbre_osm_m",
        "taux_ssmsi_vols_vehicules_commune"
    )
    .orderBy("rang")
    .limit(10)
)
